# Zero-Touch Vulnerability Remediation — Workflow Demo

Demonstrates the **post-normalisation enrichment pipeline**.

## How it works

```
[Normalisation Agent]           <- Not shown here
        |  df.to_csv(path)      <- Normalised agent saves CSV
        v
[normalized_output.csv]         <- Hand-off point (this demo starts here!)
        |  run_remediation_pipeline(csv_file_path)
        v
[Vulnerability Agent]           <- NVD API v2.0  -> CSV + RDS upsert
        |
        v
[Vuln Intel Agent]              <- EPSS / KEV / Exploit-DB -> CSV + RDS upsert
        |
        v
[Enriched CSV + RDS updated]
```

> The coordinator only needs **one call**: `run_remediation_pipeline(csv_file_path)`

In [1]:
import os
import sys
import pandas as pd

sys.path.append(os.path.abspath('.'))

from main_workflow import run_remediation_pipeline
from db_connection import get_db_session
from sqlalchemy import text
from prioritization_agent import run_prioritization_agent

# Show ALL data — no truncation anywhere
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)

ModuleNotFoundError: No module named 'psycopg2'

## Step 1 — Input the Path to your Normalised CSV

The normalisation agent has already run and saved its output DataFrame to a CSV file.  
Run the cell below and type the path to that CSV file when prompted.

In [ ]:
NORMALIZED_CSV_PATH = input("Enter the path to your normalized CSV file (e.g., Parsed Outputs/normalized_output.csv): ").strip()

if not NORMALIZED_CSV_PATH:
    NORMALIZED_CSV_PATH = 'Parsed Outputs/normalized_output.csv'  # default fallback
    print(f"No input provided. Defaulting to: {NORMALIZED_CSV_PATH}")

assert os.path.exists(NORMALIZED_CSV_PATH), (
    f'CSV not found: {NORMALIZED_CSV_PATH}\n'
    'Run the normalisation agent first and ensure it saves its DataFrame '
    'to this path via df.to_csv(path, index=False)'
)

print(f"\nTarget CSV found: {NORMALIZED_CSV_PATH}")

## Step 2 — Run the pipeline (single call)

One function call triggers the full LangGraph workflow:

`Vulnerability Agent` → `Vuln Intel Agent`

In [ ]:
final_state = run_remediation_pipeline(NORMALIZED_CSV_PATH)
print('\nPipeline finished. Final state:', final_state)

## Step 3 — Inspect enriched CSV

The pipeline has updated the CSV file with new columns for NVD Data, EPSS scores, KEV flags, and Exploit counts.

In [ ]:
enriched_df = pd.read_csv(NORMALIZED_CSV_PATH)
print(f'Shape: {enriched_df.shape}')
print(f'Columns: {list(enriched_df.columns)}')
enriched_df   # Jupyter renders full scrollable table

## Step 4 — Verify AWS RDS PostgreSQL

The pipeline has also successfully performed a batch upsert to the remote database.

In [ ]:
session = get_db_session()
try:
    vuln_ids = enriched_df['vuln_id'].dropna().unique().tolist()

    print('=' * 60)
    print('TABLE: vulnerabilities')
    print('=' * 60)
    rows = session.execute(
        text('SELECT vuln_id, title, description, cvss_score, cvss_vector, '
             'fix_version, published_date FROM vulnerabilities '
             'WHERE vuln_id = ANY(:ids)'),
        {'ids': vuln_ids}
    ).fetchall()
    for r in rows:
        print(f'\nID:          {r.vuln_id}')
        print(f'Title:       {r.title}')
        # Truncating description in print to avoid massive text blocks, but the full description is in the DB
        print(f'Description: {str(r.description)[:150]}...')
        print(f'CVSS Score:  {r.cvss_score}')
        print(f'CVSS Vector: {r.cvss_vector}')
        print(f'Fix Version: {r.fix_version}')
        print(f'Published:   {r.published_date}')

    print('\n' + '=' * 60)
    print('TABLE: vulnerability_intel')
    print('=' * 60)
    rows_intel = session.execute(
        text('SELECT vuln_id, epss_score, kev_flag, exploit_exists, '
             'exploit_count, last_updated FROM vulnerability_intel '
             'WHERE vuln_id = ANY(:ids)'),
        {'ids': vuln_ids}
    ).fetchall()
    for r in rows_intel:
        print(f'\nID:             {r.vuln_id}')
        print(f'EPSS Score:     {r.epss_score}')
        print(f'KEV Flag:       {r.kev_flag}')
        print(f'Exploit Exists: {r.exploit_exists}')
        print(f'Exploit Count:  {r.exploit_count}')
        print(f'Last Updated:   {r.last_updated}')
finally:
    session.close()

In [ ]:
run_prioritization_agent("final_working.csv")